In [2]:
import pandas as pd
import numpy as np

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

print("All datasets loaded")

All datasets loaded


In [3]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Date columns converted")
print(orders.dtypes)

Date columns converted
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [4]:
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

print("Original orders:", len(orders))
print("Delivered orders:", len(orders_delivered))
print("Dropped:", len(orders) - len(orders_delivered))

Original orders: 99441
Delivered orders: 96478
Dropped: 2963


In [5]:
df = orders_delivered.merge(customers, on='customer_id', how='left')
df = df.merge(order_items, on='order_id', how='left')
df = df.merge(payments, on='order_id', how='left')
df = df.merge(products, on='product_id', how='left')
df = df.merge(sellers, on='seller_id', how='left')

print("Master dataframe shape:", df.shape)
print("Columns:", df.columns.tolist())

Master dataframe shape: (115038, 33)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [6]:
df['order_year'] = df['order_purchase_timestamp'].dt.year
df['order_month'] = df['order_purchase_timestamp'].dt.month
df['order_day_of_week'] = df['order_purchase_timestamp'].dt.day_name()
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

print("New columns added")
print(df[['order_purchase_timestamp','order_year','order_month','order_day_of_week','delivery_days']].head())

New columns added
  order_purchase_timestamp  order_year  order_month order_day_of_week  \
0      2017-10-02 10:56:33        2017           10            Monday   
1      2017-10-02 10:56:33        2017           10            Monday   
2      2017-10-02 10:56:33        2017           10            Monday   
3      2018-07-24 20:41:37        2018            7           Tuesday   
4      2018-08-08 08:38:49        2018            8         Wednesday   

   delivery_days  
0            8.0  
1            8.0  
2            8.0  
3           13.0  
4            9.0  


In [7]:
# Fill missing product category with 'unknown'
df['product_category_name'] = df['product_category_name'].fillna('unknown')

# Drop rows where price is missing
df = df.dropna(subset=['price'])

# Save to processed folder
df.to_csv("../data/processed/master_df.csv", index=False)

print("Cleaned data saved!")
print("Final shape:", df.shape)

Cleaned data saved!
Final shape: (115038, 37)


In [8]:
df['order_year'] = df['order_purchase_timestamp'].dt.year
df['order_month'] = df['order_purchase_timestamp'].dt.month
df['order_day_of_week'] = df['order_purchase_timestamp'].dt.day_name()
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

print("New columns added")
print(df[['order_purchase_timestamp', 'order_year', 'order_month', 'order_day_of_week', 'delivery_days']].head())

New columns added
  order_purchase_timestamp  order_year  order_month order_day_of_week  \
0      2017-10-02 10:56:33        2017           10            Monday   
1      2017-10-02 10:56:33        2017           10            Monday   
2      2017-10-02 10:56:33        2017           10            Monday   
3      2018-07-24 20:41:37        2018            7           Tuesday   
4      2018-08-08 08:38:49        2018            8         Wednesday   

   delivery_days  
0            8.0  
1            8.0  
2            8.0  
3           13.0  
4            9.0  


In [9]:
df.to_csv("../data/processed/olist_cleaned.csv", index=False)

print("Cleaned data saved!")
print("Final shape:", df.shape)

Cleaned data saved!
Final shape: (115038, 37)
